# Tdqual 2-Dimensional characterization

In this document, we study the fundamental behavior of the TDQUAL project algorithm by using it to identify which subsets are selected under different metrics based on the $L^1$ distance.


## Library Imports

We begin by importing the libraries that will be used throughout this experiment.

In [ ]:
import random
from itertools import combinations
import inspect
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import pandas as pd

from scipy.special import comb
import scipy.spatial.distance as dist
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.sparse import coo_matrix

## TDQUAL Functions

To ensure independence from the TDQUAL library, all relevant functions are defined in the following cell.

In [ ]:
def subsetZ_stratified(Z, y, size, random_state=None):
    """Selects a stratified random subset of size `size` from the input data, preserving the class distribution of the target labels.

    Returns a stratified subset `X` of `Z` and the corresponding indices with respect to the full dataset `Z`.
    """
    rng = np.random.default_rng(random_state)

    classes, counts = np.unique(y, return_counts=True)
    proportions = counts / len(y)

    selected_idx = []

    for c, p in zip(classes, proportions):
        class_idx = np.where(y == c)[0]
        n_c = int(round(size * p))

        chosen = rng.choice(class_idx, size=n_c, replace=False)
        selected_idx.append(chosen)

    selected_idx = np.concatenate(selected_idx)
    rng.shuffle(selected_idx)

    X = Z[selected_idx]
    y_sub = y[selected_idx]

    return X, y_sub, selected_idx

def reorder_subset(Z, labels, idx, label=None):
    """
    Reorders a dataset by moving a selected subset of indices to the front while keeping the remaining samples in their original relative order.
    Optionally filters the dataset by a specific label before performing the reordering, updating the indices accordingly.

    Returns the reordered dataset, reordered labels, and the reordered subset corresponding to the selected indices.
    """
    idx = np.array(idx)

    if label is not None:

        label_mask = labels == label

        valid_indices = np.where(label_mask)[0]

        Z = Z[label_mask]
        labels = labels[label_mask]

        idx = idx[np.isin(idx, valid_indices)]

        mapping = {
            old: new
            for new, old in enumerate(valid_indices)
        }

        idx = np.array([mapping[i] for i in idx])

    mask = np.ones(len(Z), dtype=bool)
    mask[idx] = False

    Z_ord = np.vstack((Z[idx], Z[mask]))
    labels_ord = np.concatenate((labels[idx], labels[mask]))

    X_ord = Z_ord[:len(idx)]

    idx_new = np.arange(len(idx))

    return Z_ord, labels_ord, X_ord

# --------------------------------------------

def mst_edge_filtration(points):
    """Returns the edges and filtration values for the Euclidean minimum spanning tree
    of a given point sample.
    This is a wrapper for the scipy minimum_spanning_tree function.
    """
    d = dist.pdist(points)
    n = len(points)
    rows, cols = np.triu_indices(n, k=1)

    graph = coo_matrix(
        (d, (rows, cols)),
        shape=(n, n)
    )

    graph = graph + graph.T

    mst = minimum_spanning_tree(graph)

    return read_csr_matrix(mst)

def read_csr_matrix(cs_matrix):
    """Function to read output from minimum_spanning_tree and prepare it as a list of
    filtration values (in order) together with an array with the corresponding pairs.
    """
    rows, cols = cs_matrix.nonzero()

    weights = cs_matrix.data

    sort_idx = np.argsort(weights)

    edges = np.column_stack((rows, cols))[sort_idx].astype(np.int64)
    filtration_list = weights[sort_idx]

    return filtration_list.tolist(), edges

def compute_tmt_pairs_history(
    filtration_list,
    edges_arr,
    tolerance=1e-8
):
  """Computes the complete merge history of a Topological Merger Tree (TMT) from a filtration and a list of edges.
  Returns the sequence of merged component pairs, the corresponding merge sizes, the full size history, and the filtration values at which merges occur.
  """
  n = edges_arr.max() + 1

  # labels de componentes
  C = np.arange(n, dtype=np.int64)

  # tamaños de componentes
  sizes = np.ones(n, dtype=np.int64)

  tmt_pairs = []
  merge_sizes = []

  # historial COMPLETO
  size_history = [sizes.copy()]

  # estado inicial
  filtration_history = [0.0]

  E_b = []

  for i, (b, edge) in enumerate(zip(filtration_list, edges_arr)):

      E_b.append(edge)

      # acumular mismo nivel
      if (
          i < len(filtration_list) - 1
          and abs(b - filtration_list[i + 1]) < tolerance
      ):
          continue

      while E_b:

          mins = np.array([
              min(C[u], C[v])
              for u, v in E_b
          ])

          idx = np.argmin(mins)

          u, v = E_b.pop(idx)

          cu, cv = C[u], C[v]

          M = max(cu, cv)
          m = min(cu, cv)

          # ya conectados
          if M == m:
              continue

          # tamaños antes del merge
          size_M = sizes[M]
          size_m = sizes[m]

          merge_sizes.append([size_M, size_m])

          # merge TMT
          tmt_pairs.append([M, m])

          # merge labels
          C[C == M] = m

          # actualizar tamaños
          sizes[m] += sizes[M]
          sizes[M] = 0

          # snapshot tras merge
          size_history.append(sizes.copy())

          filtration_history.append(b)

  return (
      np.array(tmt_pairs, dtype=np.int64),
      np.array(merge_sizes, dtype=np.int64),
      np.array(size_history, dtype=np.int64),
      np.array(filtration_history)
  )

def get_inclusion_matrix(pairs_arr_X, pairs_arr_Z, subset_indices=[]):
    """ Given two pairs of arrays with the vertex merge pairs, this function returns the associated inclusion matrix.
    From the point of view of minimum spanning trees, the output matrix columns can be interpreted as the minimum paths that are needed to
    go through in MST(Z) in order to connect the endpoints from an edge in MST(X)
    """
    # If subset indices are not specified, we assume that the indices of vertices from S correspond to the first #X vertices from Z
    if (len(subset_indices)==0):
        subset_indices = list(range(pairs_arr_X.shape[0]+1))
    pivot2column = [-1] + np.argsort(np.max(pairs_arr_Z, axis=1)).tolist()
    inclusion_matrix = []
    for col_X in pairs_arr_X:
        col_X = [subset_indices[i] for i in col_X]
        col_M = []
        while(len(col_X)>0):
            piv = np.max(col_X)
            col_M.append(pivot2column[piv])
            col_X = add_columns_mod_2(col_X, pairs_arr_Z[pivot2column[piv]])
        # end reducing column X
        col_M.sort()
        inclusion_matrix.append(col_M)
    return inclusion_matrix

def add_columns_mod_2(col1, col2):
    """ Given two lists of integers, which are sparse representations of a pair of vectors in Z mod 2, this funciton adds them and
    returns the result in the same input format.
    """
    return list(set(col1) ^ set(col2)) #La operación ^ entre sets da los elementos que están en cada conjunto pero no en ambos


def get_inclusion_matrix_pivots(matrix_list, num_rows):
    """ Returns the pivots of a matrix given in list format"""
    pivots = []
    reduced_matrix = []
    pivot2column = np.full(num_rows, -1, dtype=int)
    for i, column in enumerate(matrix_list):
        reduce_column = list(column)
        piv = get_pivot(reduce_column)
        while(pivot2column[piv]>-1):
            reduce_column = add_columns_mod_2(reduce_column, reduced_matrix[pivot2column[piv]])
            piv = get_pivot(reduce_column)
            # we assume that columns are never reduced to the 0 column
        pivots.append(piv)
        reduced_matrix.append(reduce_column)
        pivot2column[piv] = i
    # end getting pivots
    return pivots


def get_pivot(nonempty_list):
  "given a non-empty list of integers, returns the pivot. GIves error otherwise."
  return int(np.max(nonempty_list))

def compute_Mf_0(X, Z, is_dist=False, track_reductions=False):
    """Compute the matching induced by the inclusion X --> Z given by sending points from X, in order, to the first points from Z.
    We assume that coordinates of X and Z are given as numpy arrays, where the number of rows from Z is greater or equal to X.
    If is_dist is True, we assume that X and Z are given as distance matrices.

    This returns a pair of lists with endpoints of intervals, that are representations of both barcodes, together with a matching between such lists given by a list of indices.
    """
    filtration_list_X, pairs_arr_X = mst_edge_filtration(X) # MST(X)
    filtration_list_Z, pairs_arr_Z = mst_edge_filtration(Z) # MST(Z)
    TMT_X_pairs, merge_sizes_X, _, _  = compute_tmt_pairs_history(filtration_list_X, pairs_arr_X)
    TMT_Z_pairs, merge_sizes_Z, _, _ = compute_tmt_pairs_history(filtration_list_Z, pairs_arr_Z)
    indices_X_Z = np.max(TMT_Z_pairs, axis=1)<X.shape[0]
    TMT_X_Z_pairs = TMT_Z_pairs[indices_X_Z]
    merge_sizes_X_Z = merge_sizes_Z[indices_X_Z]
    indices_X_Z = np.nonzero(indices_X_Z)[0]
    FX = get_inclusion_matrix(TMT_X_pairs, TMT_X_Z_pairs) # Associated matrix
    if track_reductions:
      matchingX, reductions = get_inclusion_matrix_pivots_tracking_reductions(FX, Z.shape[0]) # Matching in TMT_X_Z
      matching =[indices_X_Z[i] for i in matchingX] # Matching in all TMT_Z
      return filtration_list_X,TMT_X_pairs,merge_sizes_X ,filtration_list_Z, TMT_Z_pairs, merge_sizes_Z, TMT_X_Z_pairs, merge_sizes_X_Z, matching, reductions
    else:
      matchingX = get_inclusion_matrix_pivots(FX, Z.shape[0]) # Matching in TMT_X_Z
      matching =[indices_X_Z[i] for i in matchingX] # Matching in all TMT_Z
      return filtration_list_X,TMT_X_pairs,merge_sizes_X ,filtration_list_Z, TMT_Z_pairs, merge_sizes_Z, TMT_X_Z_pairs, merge_sizes_X_Z, matching

def compute_matching_diagram(filt_X, filt_Z, matching, tol=1e-5):
    """Computes the matching diagram between `filt_X` and `filt_Z`, grouping repeated values within a tolerance.
    Returns the matching pairs between the two filtrations and their corresponding counts.
    """
    filt_X = np.asarray(filt_X)
    filt_Z = np.asarray(filt_Z)
    matching = np.asarray(matching)

    # -----------------------------------------
    # Matched pairs
    # -----------------------------------------

    pairs = np.column_stack((
        filt_X,
        filt_Z[matching]
    ))

    # Reverse to mimic pop()
    pairs = pairs[::-1]

    diff = np.abs(np.diff(pairs, axis=0))

    same = np.all(diff < tol, axis=1)

    group_start = np.concatenate((
        [True],
        ~same
    ))

    idx = np.flatnonzero(group_start)

    unique_pairs = pairs[idx]

    counts = np.diff(np.append(idx, len(pairs)))

    # -----------------------------------------
    # Unmatched points
    # -----------------------------------------

    mask = np.ones(len(filt_Z), dtype=bool)
    mask[matching] = False

    unmatched = filt_Z[mask][::-1]

    if unmatched.size > 0:

        diff_u = np.abs(np.diff(unmatched))

        same_u = diff_u < tol

        group_start_u = np.concatenate((
            [True],
            ~same_u
        ))

        idx_u = np.flatnonzero(group_start_u)

        unique_unmatched = unmatched[idx_u]

        counts_u = np.diff(
            np.append(idx_u, len(unmatched))
        )

        unmatched_pairs = np.column_stack((
            np.full(len(unique_unmatched), np.inf),
            unique_unmatched
        ))

        unique_pairs = np.vstack((
            unique_pairs,
            unmatched_pairs
        ))

        counts = np.concatenate((
            counts,
            counts_u
        ))

    return unique_pairs, counts


def mixup_percentage_distance(points, mult):
    """Computes a mixup percentage distance between paired points.
    """

    mult = np.asarray(mult)
    _points = points[points[:,0] != np.inf]
    _mult = mult[points[:,0] != np.inf]

    x = _points[:,0]
    y = _points[:,1]

    return np.sum(_mult * np.abs((x - y)/y)).item()


def induced_matching_distance_L1(points, mult, matching_type = None):
    """Computes a weighted L1 distance for a set of matched or unmatched point pairs.
    If `matching_type` is 'matched', the distance is computed over finite pairs only.
    If it is 'unmatched', the distance is computed over pairs with infinite first coordinate, treating infinity as zero.
    If `matching_type` is None, the distance is computed over all points using the same convention for infinite values.
    """
    mult = np.asarray(mult)

    if matching_type == 'matched':

      _points = points[points[:,0] != np.inf]
      _mult = mult[points[:,0] != np.inf]

      x = _points[:,0]
      y = _points[:,1]

      return np.sum(_mult * np.abs((x - y))).item()

    elif matching_type == 'unmatched':

      _points = points[points[:,0] == np.inf]
      _mult = mult[points[:,0] == np.inf]

      x = np.where(np.isinf(_points[:, 0]), 0, _points[:, 0])
      y = _points[:,1]

      return np.sum(_mult * np.abs((x - y))).item()

    else:
      x = np.where(np.isinf(points[:, 0]), 0, points[:, 0])
      y = points[:, 1]

      return np.sum(mult * np.abs(x - y)).item()


## Two-Dimensional Applications



In [ ]:
def matching_score(Z, y, idx,label=None):
  """Computes L1 based matching scores between a subset of a dataset and its reordered version.
  """

  d_l1_matches = []
  d_l1_unmatches = []
  d_mixup = []

  Zi, yi, Xi = reorder_subset(Z, y, idx, label=label)

  filt_Xi,TMT_Xi_pairs,merge_sizes_Xi, filt_Zi, TMT_Zi_pairs,merge_sizes_Zi, TMT_Xi_Zi_pairs, merge_sizes_Xi_Zi, matchingi = compute_Mf_0(Xi,Zi)
  D_fi, mults = compute_matching_diagram(filt_Xi, filt_Zi,matchingi)

  d_l1_matches.append(induced_matching_distance_L1(D_fi, mults, matching_type='matched'))
  d_l1_unmatches.append(induced_matching_distance_L1(D_fi, mults, matching_type='unmatched'))
  d_mixup.append(mixup_percentage_distance(D_fi, mults))

  return d_l1_matches, d_l1_unmatches, d_mixup

def plot_Z(Z):
  fig, ax = plt.subplots(ncols=1, figsize=(4,4))
  ax.scatter(Z[:,0], Z[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=40, marker="X", zorder=2, label="Z")

  for j, (px, py) in enumerate(Z):
    ax.text(px + 0.025, py + 0.025, str(j), fontsize=8)

  ax.set_title('Z samples')
  plt.legend(loc='best')
  plt.show()

  return None


def plot_Z_X(ax, Z, y, idx, title, show_numbers=False):

  Zi, yi, Xi = reorder_subset(Z, y, idx, label=None)

  ax.scatter(Xi[:,0], Xi[:,1], color=mpl.colormaps["RdBu"](0.3/1.3), s=60, marker="o", zorder=3, label="X")
  ax.scatter(Zi[:,0], Zi[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=40, marker="X", zorder=2, label="Z")

  if show_numbers:
    for j, (px, py) in enumerate(Zi):
      ax.text(px + 0.025, py + 0.025, str(j), fontsize=8)

  ax.set_title(title)
  ax.legend(loc='best')


  return None

def plot_Z_X_grid(Z, y, df, show_numbers=False):
  df = df.sort_values(['Total Mixup']).reset_index(drop=True)
  points_matched_min, points_matched_max = df['points'].iloc[0], df['points'].iloc[-1]
  L1_matched_min, L1_matched_max = df['Total Mixup'].iloc[0], df['Total Mixup'].iloc[-1]

  df = df.sort_values(['Total Unmatched']).reset_index(drop=True)
  points_unmatched_min, points_unmatched_max = df['points'].iloc[0], df['points'].iloc[-1]
  L1_unmatched_min, L1_unmatched_max = df['Total Unmatched'].iloc[0], df['Total Unmatched'].iloc[-1]

  df = df.sort_values(['Total Mixup Percentage']).reset_index(drop=True)
  points_mixup_min, points_mixup_max = df['points'].iloc[0], df['points'].iloc[-1]
  L1_mixup_min, L1_mixup_max = df['Total Mixup Percentage'].iloc[0], df['Total Mixup Percentage'].iloc[-1]

  fig, axes = plt.subplots(2, 3, figsize=(12, 8))

  plot_Z_X(
      axes[0, 0],
      Z, y,
      points_matched_min,
      f"Total Mixup (Min) = {L1_matched_min:.3f}",
      show_numbers=show_numbers
  )

  plot_Z_X(
      axes[1, 0],
      Z, y,
      points_matched_max,
      f"Total Mixup (Max) = {L1_matched_max:.3f}",
      show_numbers=show_numbers
  )

  plot_Z_X(
      axes[0, 1],
      Z, y,
      points_unmatched_min,
      f"Total Unmatched (Min) = {L1_unmatched_min:.3f}",
      show_numbers=show_numbers
  )

  plot_Z_X(
      axes[1, 1],
      Z, y,
      points_unmatched_max,
      f"Total Unmatched (Max) = {L1_unmatched_max:.3f}",
      show_numbers=show_numbers
  )

  plot_Z_X(
      axes[0, 2],
      Z, y,
      points_mixup_min,
      f"Total Mixup Percentage (Min) = {L1_mixup_min:.3f}",
      show_numbers=show_numbers
  )

  plot_Z_X(
      axes[1, 2],
      Z, y,
      points_mixup_max,
      f"Total Mixup Percentage (Max) = {L1_mixup_max:.3f}",
      show_numbers=show_numbers
  )

  plt.tight_layout()
  plt.show()


### Scattered Random Points

We first consider a set of simple examples with a small number of points, progressively increasing the dataset size to examine the fundamental characteristics and implications of the different $L^1$-based distance metrics: Total Mixup (TM), Total Unmatched (TU), and Total Mixup Percentage (TMP).

#### 4 Points

Starting with four points in the set ($Z$), we can compute all possible combinations of subset selections. These are the subsets composed of two and three points.

In [ ]:
Z = np.array([[1,1.6],[4,1],[1,2],[4,1.4]])
label_Z = np.zeros(len(Z))

scores_4 = []
plot_Z(Z)

for n_points in range(2,len(Z)):
  for idxs in combinations(range(Z.shape[0]), n_points):
      score = matching_score(Z, label_Z, list(idxs))

      scores_4.append({
        "points": list(idxs),
        "Total Mixup": score[0][0],
        "Total Unmatched": score[1][0],
        "Total Mixup Percentage": score[2][0]
      })

scores_4 = pd.DataFrame(scores_4).sort_values(['Total Mixup']).reset_index(drop=True)
scores_4

By examining the DataFrame containing the distance values, it can be observed that TM and TMP are equivalent whenever the minimum value is zero.This occurs when points are selected such that the edges connecting them are identical in both ($X$) and ($Z$). Consequently, multiple subsets can be expected to attain a value of zero.

The following figure compares the subsets selected according to the extreme values of TM, TU, and TMP.


In [ ]:
plot_Z_X_grid(Z, label_Z, scores_4)

#### 5 Points

Let us now examine what happens when we add an intermediate point.

In [ ]:
Z = np.array([[1,1.8],[4,1],[1,2],[4,1.4],[2.5,1.3]])
label_Z = np.zeros(len(Z))

scores_5 = []
plot_Z(Z)

for n_points in range(2,len(Z)):
  for idxs in combinations(range(Z.shape[0]), n_points):
      score = matching_score(Z, label_Z, list(idxs))

      scores_5.append({
        "points": list(idxs),
        "Total Mixup": score[0][0],
        "Total Unmatched": score[1][0],
        "Total Mixup Percentage": score[2][0]
      })

scores_5 = pd.DataFrame(scores_5).sort_values(['Total Mixup']).reset_index(drop=True)
scores_5

We again observe multiple sets ($X$) achieving a value of 0 for both TM and TMP. This behavior appears in all examples; consequently, the grid representations show only one of the possible subsets.

In [ ]:
plot_Z_X_grid(Z, label_Z, scores_5)

#### 11 Points

We introduce additional points to form three small clusters along with a central outlier.

In [ ]:
Z = np.array([[1,1.8],[4,1],[1,2],[4,1.4],[2.5,1.3],[1.2,1.9],[4.5,1.2],[0.5,0],[1.5,0.5],[0.57,0.3],[1,0.7]])
label_Z = np.zeros(len(Z))

scores_11 = []

plot_Z(Z)

for n_points in [3,4,5]:
  for idxs in combinations(range(Z.shape[0]), n_points):
      score = matching_score(Z, label_Z, list(idxs))

      scores_11.append({
        "points": list(idxs),
        "Total Mixup": score[0][0],
        "Total Unmatched": score[1][0],
        "Total Mixup Percentage": score[2][0]
      })

scores_11 = pd.DataFrame(scores_11).sort_values(['Total Mixup']).reset_index(drop=True)
scores_11

Given the large number of combinations, we present the results for subsets of size $|X| = 5$.

In [ ]:
pick = 5
plot_Z_X_grid(Z, label_Z, scores_11[scores_11['points'].apply(len) == pick])

### Random Points Inside a Circle

#### 17 Points

We increase the size of $|Z|$ by considering a small point cloud consisting of 16 points randomly distributed within a unit circle, along with an additional central point.

In [ ]:
n_points = 16
r = 1
x0, y0 = 0, 0

rng = np.random.default_rng(42)

theta_0 = rng.uniform(0, 2*np.pi, n_points)
rho_0 = r * np.sqrt(rng.uniform(0, 1, n_points))

x = x0 + rho_0 * np.cos(theta_0)
y = y0 +rho_0 * np.sin(theta_0)

Z = [[x0,y0]]
Z = np.concatenate([Z, [[x.item(),y.item()] for x,y in zip(x,y)]])


label_Z = np.zeros(len(Z))

scores_rand_17 = []

plot_Z(Z)

for n_points in range(6,len(Z)):
  for idxs in combinations(range(Z.shape[0]), n_points):
      score = matching_score(Z, label_Z, list(idxs))

      scores_rand_17.append({
        "points": list(idxs),
        "Total Mixup": score[0][0],
        "Total Unmatched": score[1][0],
        "Total Mixup Percentage": score[2][0]
      })

scores_rand_17 = pd.DataFrame(scores_rand_17).sort_values(['Total Mixup']).reset_index(drop=True)
scores_rand_17

We observe that, for the case where $|X| = 8$, the minimum rule for TM and TMP continues to hold. In contrast, the maximum values of TM and TMP select points located in the outer region of the cloud.

Meanwhile, the selection induced by TU corresponds to regions of lower and higher density, associated with the minimum and maximum distance values, respectively.

In [ ]:
pick = 8
plot_Z_X_grid(Z, label_Z, scores_rand_17[scores_rand_17['points'].apply(len) == pick])

#### 17 Points + 1 Outlier

We now examine the behavior of multiple point clouds, with the addition of an outlier.

##### Example 1

In [ ]:
n_points = 16
r = 1
x0, y0 = 0, 0

xout, yout = 3,2

rng = np.random.default_rng(42)

theta_0 = rng.uniform(0, 2*np.pi, n_points)
rho_0 = r * np.sqrt(rng.uniform(0, 1, n_points))

x = x0 + rho_0 * np.cos(theta_0)
y = y0 +rho_0 * np.sin(theta_0)

Z = [[x0,y0],[xout,yout]]
Z = np.concatenate([Z, [[x.item(),y.item()] for x,y in zip(x,y)]])


label_Z = np.zeros(len(Z))

scores_rand_17_out = []

plot_Z(Z)

for n_points in range(6,len(Z)):
  for idxs in combinations(range(Z.shape[0]), n_points):
      score = matching_score(Z, label_Z, list(idxs))

      scores_rand_17_out.append({
        "points": list(idxs),
        "Total Mixup": score[0][0],
        "Total Unmatched": score[1][0],
        "Total Mixup Percentage": score[2][0]
      })


scores_rand_17_out = pd.DataFrame(scores_rand_17_out).sort_values(['Total Mixup']).reset_index(drop=True)
scores_rand_17_out

Returning to the possible selections with $|X| = 8$, we observe the same general behavior, although it affects the inclusion of the outlier in $X$. As the maximum values of TM and TMP tend to select outer points, the outlier is included.

In contrast, the density-driven selection of TU implies that the minimum (lower density) case includes the outlier, whereas the maximum (higher density) case does not.

In [ ]:
pick = 8
plot_Z_X_grid(Z, label_Z, scores_rand_17_out[scores_rand_17_out['points'].apply(len) == pick])

##### Example 2

When repeating this example with a different point cloud, we obtain a similar result.

In [ ]:
n_points = 16
r = 1
x0, y0 = 0, 0

xout, yout = 3,2

rng = np.random.default_rng(25)

theta_0 = rng.uniform(0.25, 2*np.pi, n_points)
rho_0 = r * np.sqrt(rng.uniform(0, 1, n_points))

x = x0 + rho_0 * np.cos(theta_0)
y = y0 +rho_0 * np.sin(theta_0)

Z = [[x0,y0],[xout,yout]]
Z = np.concatenate([Z, [[x.item(),y.item()] for x,y in zip(x,y)]])


label_Z = np.zeros(len(Z))

scores_rand_17_out_2 = []

plot_Z(Z)

for n_points in range(6,len(Z)):
  for idxs in combinations(range(Z.shape[0]), n_points):
      score = matching_score(Z, label_Z, list(idxs))

      scores_rand_17_out_2.append({
        "points": list(idxs),
        "Total Mixup": score[0][0],
        "Total Unmatched": score[1][0],
        "Total Mixup Percentage": score[2][0]
      })

scores_rand_17_out_2 = pd.DataFrame(scores_rand_17_out_2).sort_values(['Total Mixup']).reset_index(drop=True)
scores_rand_17_out_2

In [ ]:
pick = 8
plot_Z_X_grid(Z, label_Z, scores_rand_17_out_2[scores_rand_17_out_2['points'].apply(len) == pick])

##### Example 3



In [ ]:
n_points = 16
r = 1
x0, y0 = 0, 0

xout, yout = 3,2

rng = np.random.default_rng(111)

theta_0 = rng.uniform(0, 2*np.pi, n_points)
rho_0 = r * np.sqrt(rng.uniform(0, 1, n_points))

x = x0 + rho_0 * np.cos(theta_0)
y = y0 +rho_0 * np.sin(theta_0)

Z = [[x0,y0],[xout,yout]]
Z = np.concatenate([Z, [[x.item(),y.item()] for x,y in zip(x,y)]])


label_Z = np.zeros(len(Z))

scores_rand_17_out_3 = []

plot_Z(Z)

pick = 8

print(f'{comb(len(Z), pick, exact=True)} combinations...')

for idxs in combinations(range(Z.shape[0]), pick):
  score = matching_score(Z, label_Z, list(idxs))

  scores_rand_17_out_3.append({
    "points": list(idxs),
    "Total Mixup": score[0][0],
    "Total Unmatched": score[1][0],
    "Total Mixup Percentage": score[2][0]
  })


scores_rand_17_out_3 = pd.DataFrame(scores_rand_17_out_3).sort_values(['Total Mixup']).reset_index(drop=True)
scores_rand_17_out_3

In [ ]:
plot_Z_X_grid(Z, label_Z, scores_rand_17_out_3, show_numbers=False)

### Random Points Inside two Circles

#### 21 Points + 1 Outlier

Finally, we study the most complex scenario in this notebook by simulating two clusters and one outlier in a set $Z$ of size 21.

We restrict the computation to combinations where $|X| = 9$.

##### Example 1

In [ ]:
n_points = 9
r = 1
x0, y0 = 0, 0

xout, yout = 4,2

rng = np.random.default_rng(122)

theta_0 = rng.uniform(0, 2*np.pi, n_points)
rho_0 = r * np.sqrt(rng.uniform(0, 1, n_points))

x = x0 + rho_0 * np.cos(theta_0)
y = y0 +rho_0 * np.sin(theta_0)

Z = [[x0,y0],[xout,yout]]
Z = np.concatenate([Z, [[x.item(),y.item()] for x,y in zip(x,y)]])


r = 1
x0, y0 = 3, -3

rng = np.random.default_rng(211)

theta_0 = rng.uniform(0, 2*np.pi, n_points)
rho_0 = r * np.sqrt(rng.uniform(0, 1, n_points))

x = x0 + rho_0 * np.cos(theta_0)
y = y0 +rho_0 * np.sin(theta_0)


Z = np.vstack([Z,[x0,y0]])
Z = np.concatenate([Z, [[x.item(),y.item()] for x,y in zip(x,y)]])



label_Z = np.zeros(len(Z))

scores_rand_2_22 = []

plot_Z(Z)

pick = 9

print(f'{comb(len(Z), pick, exact=True)} combinations...')

for idxs in combinations(range(Z.shape[0]), pick):
  score = matching_score(Z, label_Z, list(idxs))

  scores_rand_2_22.append({
    "points": list(idxs),
    "Total Mixup": score[0][0],
    "Total Unmatched": score[1][0],
    "Total Mixup Percentage": score[2][0]
  })

scores_rand_2_22 = pd.DataFrame(scores_rand_2_22).sort_values(['Total Mixup']).reset_index(drop=True)
scores_rand_2_22

Minimising TM and TMP continues to select points that tend to be arranged such that no points of Z∖X lie between them.

The minimisation of TU continues to favour the selection of outer points from each cluster, with a higher proportion of samples drawn from the less dense cluster and including the outlier.

The maximisation of TM and TMP exhibits a configuration similar to that obtained by the minimisation of TU: both criteria select the outlier together with outer points from the clusters.

Finally, the maximisation of TU, driven by density, concentrates the selection on points belonging to a single cluster, similarly to the selection pattern previously observed for the minimisation of TMP.

In [ ]:
plot_Z_X_grid(Z, label_Z, scores_rand_2_22, show_numbers=False)

##### Example 2

The last example shows similar results.

In [ ]:
n_points = 9
r = 1
x0, y0 = 0, 0

xout, yout = 4,2

rng = np.random.default_rng(12)

theta_0 = rng.uniform(0, 2*np.pi, n_points)
rho_0 = r * np.sqrt(rng.uniform(0, 1, n_points))

x = x0 + rho_0 * np.cos(theta_0)
y = y0 +rho_0 * np.sin(theta_0)

Z = [[x0,y0],[xout,yout]]
Z = np.concatenate([Z, [[x.item(),y.item()] for x,y in zip(x,y)]])


r = 1
x0, y0 = 3, -3

rng = np.random.default_rng(21)

theta_0 = rng.uniform(0, 2*np.pi, n_points)
rho_0 = r * np.sqrt(rng.uniform(0, 1, n_points))

x = x0 + rho_0 * np.cos(theta_0)
y = y0 +rho_0 * np.sin(theta_0)


Z = np.vstack([Z,[x0,y0]])
Z = np.concatenate([Z, [[x.item(),y.item()] for x,y in zip(x,y)]])



label_Z = np.zeros(len(Z))

scores_rand_2_22_2 = []

plot_Z(Z)

pick = 9

print(f'{comb(len(Z), pick, exact=True)} combinations...')

for idxs in combinations(range(Z.shape[0]), pick):
  score = matching_score(Z, label_Z, list(idxs))

  scores_rand_2_22_2.append({
    "points": list(idxs),
    "Total Mixup": score[0][0],
    "Total Unmatched": score[1][0],
    "Total Mixup Percentage": score[2][0]
  })

scores_rand_2_22_2 = pd.DataFrame(scores_rand_2_22_2).sort_values(['Total Mixup']).reset_index(drop=True)
scores_rand_2_22_2

In [ ]:
plot_Z_X_grid(Z, label_Z, scores_rand_2_22_2, show_numbers=False)